In [1]:
import pandas as pd
import numpy as np

In [10]:
df_base = pd.read_parquet('../../data/gold/forecasting/ds_base_dataset.parquet')
df_ts = pd.read_parquet('../../data/gold/data_warehouse/dw_dim_time_series.parquet')

In [5]:
df_base.columns

Index(['time_series_id', 'week_date', 'supplier_id', 'region_id', 'product_id',
       'units_sold', 'week', 'start_date', 'end_date', 'year', 'semester',
       'semester_date', 'semester_name', 'quarter', 'quarter_date',
       'quarter_name', 'month', 'month_name', 'month_date', 'first_week_date',
       'last_week_date', 'total_weeks_length', 'num_week_with_sales',
       'num_week_with_zeros', 'sales_weeks_ratio', 'sales_units',
       'avg_weekly_sales', 'avg_weekly_sales_non_zero', 'std_weekly_sales',
       'std_weekly_sales_non_zero', 'max_weekly_sales', 'min_weekly_sales',
       'q25_sales', 'q50_sales', 'q75_sales', 'iqr', 'cv', 'supplier_name',
       'region_name', 'product_name', 'product_attribute_1',
       'product_attribute_2', 'product_attribute_3'],
      dtype='object')

In [8]:
# =========================================================================
# SOLUÇÃO CORRETA - Spine Individual + Referência Global
# =========================================================================

def segment_by_maturity_with_global_reference(
    df_base,
    df_ts_metadata,
    reference_date=None  # "Hoje" (data de referência)
):
    """
    Segmenta séries usando:
    - Spine INDIVIDUAL (eficiente)
    - Referência GLOBAL (para detectar descontinuadas)
    
    Parameters:
    -----------
    reference_date : date
        Data de referência ("hoje")
        Se None, usa max(week_date) do dataset
    """
    
    print("📊 SEGMENTAÇÃO POR MATURIDADE")
    print("="*70)
    
    # =========================================================================
    # PASSO 1: Definir data de referência ("HOJE")
    # =========================================================================
    
    if reference_date is None:
        reference_date = df_base['week_date'].max()
    
    print(f"\n   📅 Data de referência ('hoje'): {reference_date}")
    
    # Converter para datetime se necessário
    reference_date = pd.to_datetime(reference_date)
    
    # Data limite para considerar descontinuada (26 semanas atrás)
    discontinued_threshold = reference_date - pd.Timedelta(weeks=26)
    
    print(f"   📅 Threshold descontinuada: {discontinued_threshold}")
    print(f"      (séries inativas após essa data = descontinuadas)")
    
    # =========================================================================
    # PASSO 2: Classificar cada série
    # =========================================================================
    
    df_ts = df_ts_metadata.copy()
    df_ts['segment'] = 'unknown'
    df_ts['reference_date'] = reference_date
    
    # Converter last_week_date para datetime
    df_ts['last_week_date'] = pd.to_datetime(df_ts['last_week_date'])
    
    # Calcular semanas desde última venda
    df_ts['weeks_since_last_sale'] = (
        (reference_date - df_ts['last_week_date']).dt.days / 7
    ).astype(int)
    
    # -------------------------------------------------------------------------
    # GRUPO 3: DESCONTINUADAS (verificar PRIMEIRO!)
    # -------------------------------------------------------------------------
    
    discontinued_criteria = (
        (df_ts['num_week_with_sales'] == 0) |  # Nunca vendeu
        (df_ts['last_week_date'] < discontinued_threshold)  # Inativa há >26 semanas
    )
    
    df_ts.loc[discontinued_criteria, 'segment'] = 'discontinued'
    
    n_discontinued = discontinued_criteria.sum()
    
    print(f"\n   ❌ DESCONTINUADAS: {n_discontinued:,} séries")
    
    # Subcategorias
    never_sold = (df_ts['num_week_with_sales'] == 0).sum()
    inactive = (
        (df_ts['num_week_with_sales'] > 0) &
        (df_ts['last_week_date'] < discontinued_threshold)
    ).sum()
    
    print(f"      Nunca vendeu: {never_sold:,}")
    print(f"      Inativa >26 semanas: {inactive:,}")
    
    # Mostrar exemplos
    if inactive > 0:
        examples = df_ts[
            (df_ts['num_week_with_sales'] > 0) &
            (df_ts['last_week_date'] < discontinued_threshold)
        ].nsmallest(5, 'last_week_date')
        
        print(f"\n      Exemplos de inativas:")
        for _, row in examples.iterrows():
            print(f"         Série {row['time_series_id']}: "
                  f"última venda em {row['last_week_date'].date()} "
                  f"({row['weeks_since_last_sale']} semanas atrás)")
    
    # -------------------------------------------------------------------------
    # GRUPO 1: VÁLIDAS (dados suficientes E ativas)
    # -------------------------------------------------------------------------
    
    valid_criteria = (
        (df_ts['segment'] != 'discontinued') &  # Não descontinuada
        (df_ts['total_weeks_length'] >= 4) &    # Histórico suficiente
        (df_ts['num_week_with_sales'] >= 2) &   # Vendas suficientes
        (df_ts['sales_weeks_ratio'] >= 0.02)    # Fill rate mínimo
    )
    
    df_ts.loc[valid_criteria, 'segment'] = 'valid'
    
    n_valid = valid_criteria.sum()
    
    print(f"\n   ✅ VÁLIDAS: {n_valid:,} séries")
    print(f"      Critérios: weeks≥4 AND sales≥2 AND fill_rate≥2% AND ativa")
    
    # -------------------------------------------------------------------------
    # GRUPO 2: COLD START (resto)
    # -------------------------------------------------------------------------
    
    cold_start_mask = (df_ts['segment'] == 'unknown')
    df_ts.loc[cold_start_mask, 'segment'] = 'cold_start'
    
    n_cold_start = cold_start_mask.sum()
    
    print(f"\n   ⚠️  COLD START: {n_cold_start:,} séries")
    
    # Subcategorias
    print(f"      Subcategorias:")
    
    new_product = (
        (df_ts['segment'] == 'cold_start') &
        (df_ts['total_weeks_length'] < 4)
    ).sum()
    print(f"         Produto novo (<4 semanas): {new_product:,}")
    
    single_sale = (
        (df_ts['segment'] == 'cold_start') &
        (df_ts['num_week_with_sales'] == 1)
    ).sum()
    print(f"         Uma única venda: {single_sale:,}")
    
    ultra_sparse = (
        (df_ts['segment'] == 'cold_start') &
        (df_ts['sales_weeks_ratio'] < 0.02)
    ).sum()
    print(f"         Ultra-esparso (<2% fill): {ultra_sparse:,}")
    
    # =========================================================================
    # PASSO 3: Separar datasets
    # =========================================================================
    
    segments = {}
    
    for segment in ['valid', 'cold_start', 'discontinued']:
        
        series_ids = df_ts[df_ts['segment'] == segment]['time_series_id'].values
        
        df_segment = df_base[
            df_base['time_series_id'].isin(series_ids)
        ].copy()
        
        # Adicionar informação de segmento
        df_segment = df_segment.merge(
            df_ts[['time_series_id', 'segment', 'weeks_since_last_sale']],
            on='time_series_id',
            how='left'
        )
        
        segments[segment] = df_segment
    
    # =========================================================================
    # PASSO 4: Validação
    # =========================================================================
    
    print("\n" + "="*70)
    print("📊 RESUMO DA SEGMENTAÇÃO")
    print("="*70)
    
    total = len(df_ts)
    
    summary = pd.DataFrame({
        'Segmento': ['Válidas', 'Cold Start', 'Descontinuadas', 'TOTAL'],
        'N_Séries': [n_valid, n_cold_start, n_discontinued, total],
        'Percentual': [
            f"{n_valid/total*100:.1f}%",
            f"{n_cold_start/total*100:.1f}%",
            f"{n_discontinued/total*100:.1f}%",
            "100.0%"
        ],
        'Estratégia': [
            'Modelagem Direta',
            'Similaridade',
            'Forecast = 0',
            '-'
        ]
    })
    
    print(f"\n{summary.to_string(index=False)}")
    
    # Verificar
    assert n_valid + n_cold_start + n_discontinued == total, "Erro: séries perdidas!"
    
    print(f"\n✅ Validação: {n_valid + n_cold_start + n_discontinued} = {total}")
    
    return segments, df_ts


# =========================================================================
# USO
# =========================================================================

# Opção 1: Usar max_date do dataset (padrão)
segments, df_ts_segmented = segment_by_maturity_with_global_reference(
    df_base,
    df_ts,
    reference_date=None  # Usa max(week_date) automaticamente
)

📊 SEGMENTAÇÃO POR MATURIDADE

   📅 Data de referência ('hoje'): 2024-10-21 00:00:00
   📅 Threshold descontinuada: 2024-04-22 00:00:00
      (séries inativas após essa data = descontinuadas)

   ❌ DESCONTINUADAS: 555 séries
      Nunca vendeu: 0
      Inativa >26 semanas: 555

      Exemplos de inativas:
         Série 15: última venda em 2022-10-31 (103 semanas atrás)
         Série 66: última venda em 2022-10-31 (103 semanas atrás)
         Série 130: última venda em 2022-10-31 (103 semanas atrás)
         Série 235: última venda em 2022-10-31 (103 semanas atrás)
         Série 246: última venda em 2022-10-31 (103 semanas atrás)

   ✅ VÁLIDAS: 2,232 séries
      Critérios: weeks≥4 AND sales≥2 AND fill_rate≥2% AND ativa

   ⚠️  COLD START: 54 séries
      Subcategorias:
         Produto novo (<4 semanas): 52
         Uma única venda: 38
         Ultra-esparso (<2% fill): 2

📊 RESUMO DA SEGMENTAÇÃO

      Segmento  N_Séries Percentual       Estratégia
       Válidas      2232      78.6%

In [12]:
df_ts_segmented[['time_series_id', 'segment']].to_parquet('../../data/gold/forecasting/ds_segmented_time_series.parquet', index=False)